# Metrik Toplayıcı

Google Drive'daki tüm eğitim koşularının metrik dosyalarını tarar, tek bir
özet tablo üretir ve hepsini tek bir `.zip` dosyasında toplar.

Toplanan dosyalar:

| Dosya | İçerik |
|---|---|
| `results.csv` | devir devir kayıp ve metrik geçmişi |
| `args.yaml` | o koşunun tüm hiperparametreleri |
| `*.json` | özet metrikler (mAP, kesinlik, duyarlılık) |
| `confusion_matrix*.png` | karışıklık matrisleri |
| `*_curve.png` | PR, F1, P, R eğrileri |
| `results.png` | eğitim eğrileri |

Model ağırlıkları (`.pt`) **dâhil edilmez** — boyut nedeniyle.

**Kullanım:** hücreleri sırayla çalıştırın. Son hücre zip dosyasını indirir.

---
*Sigorta Sektöründe Derin Öğrenme Tabanlı Otomatik Araç Hasar Tespit Sistemi*
*Vesalettin Burhan Özdemir — Üsküdar Üniversitesi, 2026*

## 1. Drive'ı bağla ve dizinleri ayarla

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os, re, json, glob, shutil, zipfile
from datetime import datetime

# ---- Dizinler (kendi yapınıza göre düzenleyin) --------------------------
ANA_DIZIN    = '/content/drive/MyDrive/tez_arac_hasar'
MODEL_DIZIN  = os.path.join(ANA_DIZIN, 'modeller')
CIKTI_DIZIN  = os.path.join(ANA_DIZIN, 'ciktilar')

# Toplanacak dosya desenleri
DESENLER = [
    'results.csv', 'args.yaml', '*.json',
    'confusion_matrix*.png', '*_curve.png', 'results.png',
    'labels*.jpg', 'val_batch*_pred.jpg',
]

# Asla dâhil edilmeyecekler
HARIC = ('.pt', '.pth', '.onnx', '.engine', '.zip')

ZAMAN = datetime.now().strftime('%Y%m%d_%H%M')
GECICI = f'/content/metrikler_{ZAMAN}'
ZIP_YOLU = f'/content/metrikler_{ZAMAN}.zip'

assert os.path.isdir(MODEL_DIZIN), f'Bulunamadi: {MODEL_DIZIN}'
print('Model dizini :', MODEL_DIZIN)
print('Cikti zip    :', ZIP_YOLU)

## 2. Koşuları tara

Her alt klasör bir eğitim koşusudur.

In [ ]:
kosular = sorted(
    d for d in os.listdir(MODEL_DIZIN)
    if os.path.isdir(os.path.join(MODEL_DIZIN, d))
)

print(f'{len(kosular)} klasor bulundu:\n')
envanter = {}

for k in kosular:
    kok = os.path.join(MODEL_DIZIN, k)
    bulunan = []
    for desen in DESENLER:
        for yol in glob.glob(os.path.join(kok, '**', desen), recursive=True):
            if yol.lower().endswith(HARIC):
                continue
            bulunan.append(yol)
    bulunan = sorted(set(bulunan))
    envanter[k] = bulunan
    isaret = 'OK ' if bulunan else '-- '
    print(f'  {isaret}{k:<48} {len(bulunan):>3} dosya')

toplam = sum(len(v) for v in envanter.values())
print(f'\nToplam {toplam} metrik dosyasi')

## 3. Özet tablo

`results.csv` dosyalarının son satırından en iyi devir metriklerini çıkarır.

In [ ]:
import pandas as pd

def en_iyi_satir(csv_yolu):
    """results.csv icinden maske mAP50'si en yuksek devri dondurur."""
    try:
        df = pd.read_csv(csv_yolu)
    except Exception as e:
        return None, f'okunamadi: {e}'
    df.columns = [c.strip() for c in df.columns]
    hedef = next((c for c in df.columns if 'mAP50(M)' in c), None)
    if hedef is None:
        hedef = next((c for c in df.columns if 'mAP50(B)' in c), None)
    if hedef is None:
        return None, 'metrik sutunu yok'
    df = df.dropna(subset=[hedef])
    if df.empty:
        return None, 'bos'
    return df.loc[df[hedef].idxmax()], None


satirlar = []
for kosu, dosyalar in envanter.items():
    csv_yolu = next((d for d in dosyalar if d.endswith('results.csv')), None)
    if not csv_yolu:
        continue
    satir, hata = en_iyi_satir(csv_yolu)
    if satir is None:
        print(f'  atlandi: {kosu} ({hata})')
        continue

    def al(*adaylar):
        for a in adaylar:
            for c in satir.index:
                if a in c:
                    return round(float(satir[c]), 4)
        return None

    satirlar.append({
        'kosu'      : kosu,
        'devir'     : int(satir.get('epoch', -1)),
        'box_mAP50' : al('metrics/mAP50(B)'),
        'box_mAP5095': al('metrics/mAP50-95(B)'),
        'mask_mAP50': al('metrics/mAP50(M)'),
        'mask_mAP5095': al('metrics/mAP50-95(M)'),
        'kesinlik'  : al('metrics/precision(M)', 'metrics/precision(B)'),
        'duyarlilik': al('metrics/recall(M)', 'metrics/recall(B)'),
    })

ozet = pd.DataFrame(satirlar)
if not ozet.empty:
    ozet = ozet.sort_values('mask_mAP50', ascending=False, na_position='last')
    ozet = ozet.reset_index(drop=True)

pd.set_option('display.width', 200)
pd.set_option('display.max_columns', 20)
display(ozet)

## 4. JSON metrik dosyalarını birleştir

In [ ]:
json_ozet = {}

for kosu, dosyalar in envanter.items():
    for yol in dosyalar:
        if not yol.endswith('.json'):
            continue
        try:
            with open(yol, encoding='utf-8') as f:
                icerik = json.load(f)
        except Exception as e:
            print(f'  okunamadi: {os.path.basename(yol)} ({e})')
            continue
        anahtar = f'{kosu}/{os.path.basename(yol)}'
        json_ozet[anahtar] = icerik

print(f'{len(json_ozet)} JSON dosyasi okundu\n')
for a in sorted(json_ozet):
    alanlar = list(json_ozet[a])[:8] if isinstance(json_ozet[a], dict) else ['(liste)']
    print(f'  {a}')
    print(f'      alanlar: {alanlar}')

## 5. Hepsini tek zip'te topla

In [ ]:
if os.path.exists(GECICI):
    shutil.rmtree(GECICI)
os.makedirs(GECICI, exist_ok=True)

kopyalanan = 0
for kosu, dosyalar in envanter.items():
    if not dosyalar:
        continue
    hedef_kok = os.path.join(GECICI, kosu)
    for yol in dosyalar:
        bagil = os.path.relpath(yol, os.path.join(MODEL_DIZIN, kosu))
        hedef = os.path.join(hedef_kok, bagil)
        os.makedirs(os.path.dirname(hedef), exist_ok=True)
        shutil.copy2(yol, hedef)
        kopyalanan += 1

# Turetilen ozetleri de ekle
if not ozet.empty:
    ozet.to_csv(os.path.join(GECICI, '00_OZET_TABLO.csv'), index=False)
with open(os.path.join(GECICI, '00_TUM_JSON.json'), 'w', encoding='utf-8') as f:
    json.dump(json_ozet, f, ensure_ascii=False, indent=2)

with open(os.path.join(GECICI, '00_OKUBENI.txt'), 'w', encoding='utf-8') as f:
    f.write(
        'Otomatik Arac Hasar Tespit Sistemi - metrik arsivi\n'
        f'Olusturulma: {datetime.now():%Y-%m-%d %H:%M}\n'
        f'Kaynak      : {MODEL_DIZIN}\n'
        f'Kosu sayisi : {len([k for k,v in envanter.items() if v])}\n'
        f'Dosya sayisi: {kopyalanan}\n\n'
        'Model agirliklari (.pt) boyut nedeniyle dahil edilmemistir.\n'
    )

with zipfile.ZipFile(ZIP_YOLU, 'w', zipfile.ZIP_DEFLATED) as z:
    for kok, _, dosyalar in os.walk(GECICI):
        for d in dosyalar:
            tam = os.path.join(kok, d)
            z.write(tam, os.path.relpath(tam, GECICI))

mb = os.path.getsize(ZIP_YOLU) / 1024 / 1024
print(f'{kopyalanan} dosya kopyalandi')
print(f'Zip hazir: {ZIP_YOLU}  ({mb:.1f} MB)')

## 6. İndir ve Drive'a yedekle

In [ ]:
# Drive'a kopyala
os.makedirs(CIKTI_DIZIN, exist_ok=True)
drive_hedef = os.path.join(CIKTI_DIZIN, os.path.basename(ZIP_YOLU))
shutil.copy2(ZIP_YOLU, drive_hedef)
print('Drive:', drive_hedef)

# Tarayiciya indir
from google.colab import files
files.download(ZIP_YOLU)